# 趋势信号研究

In [12]:
import pandas as pd
import databento as db
from pathlib import Path
import numpy as np

data_dir = Path("../data/raw")

PRODUCTS = {
    "ES": {"point_value": 50.0, "commission": 2.5, "slippage_points": 0.25},   # E-mini S&P 500
    "CL": {"point_value": 1000.0, "commission": 2.5, "slippage_points": 0.02}, # Crude Oil
    "GC": {"point_value": 100.0, "commission": 2.5, "slippage_points": 0.10},  # Gold
    "ZN": {"point_value": 1000.0, "commission": 2.5, "slippage_points": 0.01}, # 10Y Treasury
}

In [13]:
# 读取原始价格数据（未做换月价差处理）
pattern = "*-continuous-*.dbn.zst"
file_paths = sorted(data_dir.glob(pattern))
products = {}
for file_path in file_paths:
    product_name = file_path.name.split("-")[0]
    store = db.DBNStore.from_file(file_path)
    df = store.to_df()
    df = df[["instrument_id", "open", "high", "low", "close", "volume"]]
    df = df.dropna(subset=["close"])
    df = df[~df.index.duplicated(keep="last")]
    raw = df
    products[product_name] = raw



In [ ]:
products

## 单信号

### 简单时序动量因子

In [ ]:
# 简单时序动量因子 - 动量因子 = 过去 N 天的累积收益率
# 简单时序动量因子 - 过去 N 天累积收益率为正则做多，为负则做空

es_daily = products['ES']
es_params = PRODUCTS['ES']

look_back = 20

# 使用比例调整价差
is_roll = es_daily["instrument_id"] != es_daily["instrument_id"].shift(1)
is_roll.iloc[0] = False

prev_close = es_daily["close"].shift(1)
roll_ratio = prev_close / es_daily["open"]

factors = np.ones(len(es_daily))
cumulative_factor = 1.0
for i in range(len(es_daily) - 1, 0, -1):
    if is_roll.iloc[i]:
        cumulative_factor *= roll_ratio.iloc[i]
    factors[i - 1] = cumulative_factor

if len(es_daily) > 0:
    factors[0] = cumulative_factor

adjusted = es_daily[["open", "high", "low", "close", "volume"]].copy()
for col in ["open", "high", "low", "close"]:
    adjusted[col] = adjusted[col] * factors


In [23]:
close = adjusted["close"]
returns = close.pct_change(look_back)
signal = pd.Series(0.0, index=close.index)
signal[returns > 0] = 1.0
signal[returns < 0] = -1.0
signal

ts_event
2019-01-01 00:00:00+00:00    0.0
2019-01-02 00:00:00+00:00    0.0
2019-01-03 00:00:00+00:00    0.0
2019-01-04 00:00:00+00:00    0.0
2019-01-06 00:00:00+00:00    0.0
                            ... 
2026-03-25 00:00:00+00:00   -1.0
2026-03-26 00:00:00+00:00   -1.0
2026-03-27 00:00:00+00:00   -1.0
2026-03-29 00:00:00+00:00   -1.0
2026-03-30 00:00:00+00:00   -1.0
Length: 2256, dtype: float64

多动量因子组合
不同窗口等权叠加

### MACD
从MACD线 -> 趋势动量的加速度 -> 标准化

MACD 线 = EMA(fast) - EMA(slow)
反映快慢均线的距离

信号线 = EMA(MACD, signal_period)
把macd_line再做一次平滑，作为参考基准线

柱状图 = MACD - 信号线
衡量的是当前MACD line相对它自己的平滑均值，高了多少/低了多少
更像是趋势动量的加速度

histogram > 0：MACD 线在信号线上方，趋势在增强
histogram < 0：MACD 线在信号线下方，趋势在减弱或向下

计算滚动标准差
histogram/滚动标准差，从而使得不同阶段的数值可以比较
再强行压缩成一个统一的[-1,1]的信号

In [24]:
es_daily = products['ES']
es_params = PRODUCTS['ES']

# macd计算按理来说应该选用panama 价差
# Panama 调整法
# 换月时计算价差，当天open-前一天close
# 从后往前累加，是最近价格保持真实值，历史价格做偏移
# 从而保留绝对价差关系

is_roll = es_daily["instrument_id"] != es_daily["instrument_id"].shift(1)
is_roll.iloc[0] = False

overnight_gap = es_daily["open"] - es_daily["close"].shift(1)

adjustments = np.zeros(len(es_daily))
cumulative_adj = 0.0

for i in range(len(es_daily) -1, 0, -1):
    if is_roll.iloc[i]:
        cumulative_adj += overnight_gap.iloc[i]
    adjustments[i-1] = cumulative_adj

if len(es_daily) > 0:
    adjustments[0] = cumulative_adj

adjusted = es_daily[["open", "high", "low", "close", "volume"]].copy()
for col in ["open", "high", "low", "close"]:
    adjusted[col] = adjusted[col] - adjustments




In [ ]:
fast = 12
slow = 26
signal_period = 9

close = adjusted["close"]
ema_fast = close.ewm(span=fast, adjust=False).mean()
ema_slow = close.ewm(span=slow, adjust=False).mean()
macd_line = ema_fast - ema_slow
signal_line = macd_line.ewm(span=signal_period, adjust=False).mean()
histogram = macd_line - signal_line

# 计算滚动标准差
rolling_std = histogram.rolling(63, min_periods=20).std()   # 63 是季度级别的“近期常态”，3个月=63个交易日
"""
标准差的直觉是：
这些数离它们自己的平均值，平均有多分散
所以：
rolling_std 大：最近 histogram 忽上忽下，波动很大
rolling_std 小：最近 histogram 比较平稳
Pandas 这里默认是“样本标准差”，也就是分母用 n-1，不是 n。这个细节通常不会改变策略直觉，但知道它更严谨一点
"""
normalized = histogram / rolling_std.replace(0, np.nan) #当前 histogram 有多强，按“几个标准差”衡量
normalized = normalized.clip(-2, 2) / 2  # [-2σ, +2σ] → [-1, +1] 把这个强度压到固定范围，方便后面统一使用
normalized.fillna(0.0)
